In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping


In [2]:
# LOAD DATA
# =========================================
df = pd.read_csv("step9_final_dataset.csv")

df["DATE"] = pd.to_datetime(df["DATE"])
df = df.sort_values(["WELL","DATE"]).reset_index(drop=True)


In [3]:
# FEATURE ENGINEERING
# =========================================

# STATE
df["IS_ACTIVE"] = (df["HOURS"] > 0).astype(int)

# DIFF FEATURES
df["WATER_DIFF"] = df.groupby("WELL")["WATER"].diff()
df["GAS_DIFF"]   = df.groupby("WELL")["W_GAS"].diff()
df["COND_DIFF"]  = df.groupby("WELL")["COND_VOL"].diff()

df["WHP_DIFF"]   = df.groupby("WELL")["WHP"].diff()
df["WHT_DIFF"]   = df.groupby("WELL")["WHT"].diff()

# ROLLING FEATURES
df["WATER_ROLL7"] = df.groupby("WELL")["WATER"].transform(lambda x: x.rolling(7).mean())
df["GAS_ROLL7"]   = df.groupby("WELL")["W_GAS"].transform(lambda x: x.rolling(7).mean())

# CUMULATIVE FEATURES
df["CUM_WATER"] = df.groupby("WELL")["WATER"].cumsum()
df["CUM_GAS"]   = df.groupby("WELL")["W_GAS"].cumsum()

# TIME
df["MONTH"] = df["DATE"].dt.month

# OUTLIER CLIPPING
df["WATER"] = df["WATER"].clip(upper=df["WATER"].quantile(0.99))

# LOG TRANSFORM
df["WATER_LOG"] = np.log1p(df["WATER"])

# CLEAN NaN
df.fillna(0, inplace=True)


a:\pfe master\PEG_Python-master\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


,DATE,WELL,HOURS,WHP,WHT,W_GAS,WATER,COND_VOL,DELTA_DAYS,IS_ACTIVE,...,GAS_DIFF,COND_DIFF,WHP_DIFF,WHT_DIFF,WATER_ROLL7,GAS_ROLL7,CUM_WATER,CUM_GAS,MONTH,WATER_LOG
0,1999-03-29,TFT-302,3.5,100.0,47.0,0.056289,0.085220,11.448846,1.0,1,...,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.085220,0.056289,3,0.081783
1,1999-04-01,TFT-302,8.3,100.0,53.0,0.127752,0.229435,24.612087,3.0,1,...,0.071462,13.163241,0.0,6.0,0.000000,0.000000,0.314655,0.184041,4,0.206555
2,1999-04-04,TFT-302,15.8,100.0,52.0,0.243890,0.438012,46.986710,3.0,1,...,0.116138,22.374622,0.0,-1.0,0.000000,0.000000,0.752667,0.427931,4,0.363262
3,1999-04-05,TFT-302,24.0,99.0,54.0,0.371549,0.662891,71.110085,1.0,1,...,0.127660,24.123375,-1.0,2.0,0.000000,0.000000,1.415558,0.799481,4,0.508558
4,1999-04-06,TFT-302,24.0,100.0,55.0,0.371642,0.667447,71.598801,1.0,1,...,0.000092,0.488716,1.0,1.0,0.000000,0.000000,2.083004,1.171122,4,0.511294
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
485769,2003-01-27,TFY-30,24.0,102.0,62.0,0.360844,1.839730,66.318222,1.0,1,...,0.000000,0.000000,-2.0,0.0,1.839730,0.360844,2152.017832,545.006870,1,1.043709
485770,2003-01-28,TFY-30,24.0,102.0,62.0,0.360844,1.839730,66.318222,1.0,1,...,0.000000,0.000000,0.0,0.0,1.839730,0.360844,2153.857562,545.367714,1,1.043709
485771,2003-01-29,TFY-30,24.0,103.0,62.0,0.360844,1.839730,66.318222,1.0,1,...,0.000000,0.000000,1.0,0.0,1.839730,0.360844,2155.697292,545.728558,1,1.043709
485772,2003-01-30,TFY-30,24.0,104.0,62.0,0.360844,1.839730,66.318222,1.0,1,...,0.000000,0.000000,1.0,0.0,1.839730,0.360844,2157.537022,546.089402,1,1.043709


In [4]:
# FEATURES LIST
# =========================================
features = [
    "WHP", "WHT", "HOURS", "IS_ACTIVE",
    "WHP_DIFF", "WHT_DIFF",
    "GAS_DIFF", "WATER_DIFF", "COND_DIFF",
    "WATER_ROLL7", "GAS_ROLL7",
    "CUM_WATER", "CUM_GAS",
    "DELTA_DAYS", "MONTH"
]


In [5]:
# TRAIN / TEST SPLIT BY WELL
# =========================================
wellnames = df["WELL"].unique()
np.random.shuffle(wellnames)

train_wells = wellnames[:int(0.8*len(wellnames))]
test_wells  = wellnames[int(0.8*len(wellnames)):]

C:\Users\admin\AppData\Local\Temp\ipykernel_19568\1412278502.py:4: UserWarning: you are shuffling a 'ArrowStringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  np.random.shuffle(wellnames)


In [6]:
# SEQUENCE FUNCTION
# =========================================
def create_sequences(df, features, target, window=21):

    X_list, y_list = [], []

    for well in df["WELL"].unique():

        df_w = df[df["WELL"] == well].sort_values("DATE")

        X = df_w[features].values
        y = df_w[target].values

        for i in range(window, len(df_w)):

            X_list.append(X[i-window:i])
            y_list.append(y[i])

    return np.array(X_list), np.array(y_list)

In [7]:
# TRAIN FUNCTION (FINAL)
# =========================================
def train_lstm_for_target(target_name):

    print(f"\n===== TRAINING {target_name} =====")

    df_model = df.copy()

    # TARGET
    if target_name == "WATER":
        df_model["TARGET"] = df_model["WATER_LOG"]
    else:
        df_model["TARGET"] = df_model[target_name]

    # SPLIT
    df_train = df_model[df_model["WELL"].isin(train_wells)]
    df_test  = df_model[df_model["WELL"].isin(test_wells)]

    # SEQUENCES
    X_train, y_train = create_sequences(df_train, features, "TARGET")
    X_test, y_test   = create_sequences(df_test, features, "TARGET")

    # SCALE
    ns, t, f = X_train.shape
    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train.reshape(-1, f)).reshape(X_train.shape)
    X_test  = scaler.transform(X_test.reshape(-1, f)).reshape(X_test.shape)

    # MODEL
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
        Dropout(0.2),

        LSTM(32),
        Dropout(0.2),

        Dense(32, activation="relu"),
        Dense(1, activation="relu")
    ])

    model.compile(optimizer="adam", loss="mse")

    # EARLY STOPPING (FASTER)
    early_stop = EarlyStopping(patience=3, restore_best_weights=True)

    model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=20,
        batch_size=256,
        callbacks=[early_stop],
        verbose=1
    )

    # SAVE MODEL
    model.save(f"lstm_{target_name}.h5")

    # PRED
    y_pred = model.predict(X_test)

    # POST PROCESS
    if target_name == "WATER":
        y_pred = np.expm1(np.clip(y_pred, 0, 5))
        y_test = np.expm1(np.clip(y_test, 0, 5))

    # METRIC
    r2 = r2_score(y_test, y_pred)

    print(f"{target_name} R2:", r2)

    return model, r2

In [8]:
# RUN ALL MODELS
# =========================================
model_gas, r2_gas = train_lstm_for_target("W_GAS")
model_water, r2_water = train_lstm_for_target("WATER")
model_cond, r2_cond = train_lstm_for_target("COND_VOL")


===== TRAINING W_GAS =====


a:\pfe master\PEG_Python-master\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/20
1486/1486 ━━━━━━━━━━━━━━━━━━━━ 119s 77ms/step - loss: 0.0026 - val_loss: 0.0020
Epoch 2/20
1486/1486 ━━━━━━━━━━━━━━━━━━━━ 142s 77ms/step - loss: 0.0018 - val_loss: 0.0018
Epoch 3/20
1486/1486 ━━━━━━━━━━━━━━━━━━━━ 100s 68ms/step - loss: 0.0016 - val_loss: 0.0022
Epoch 4/20
1486/1486 ━━━━━━━━━━━━━━━━━━━━ 140s 66ms/step - loss: 0.0016 - val_loss: 0.0022
Epoch 5/20
1486/1486 ━━━━━━━━━━━━━━━━━━━━ 100s 67ms/step - loss: 0.0016 - val_loss: 0.0025


3238/3238 ━━━━━━━━━━━━━━━━━━━━ 22s 7ms/step
W_GAS R2: 0.9342764656289317

===== TRAINING WATER =====


a:\pfe master\PEG_Python-master\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/20
1486/1486 ━━━━━━━━━━━━━━━━━━━━ 101s 65ms/step - loss: 0.0341 - val_loss: 0.0260
Epoch 2/20
1486/1486 ━━━━━━━━━━━━━━━━━━━━ 140s 64ms/step - loss: 0.0173 - val_loss: 0.0303
Epoch 3/20
1486/1486 ━━━━━━━━━━━━━━━━━━━━ 97s 66ms/step - loss: 0.0159 - val_loss: 0.0319
Epoch 4/20
1486/1486 ━━━━━━━━━━━━━━━━━━━━ 101s 68ms/step - loss: 0.0149 - val_loss: 0.0292


3238/3238 ━━━━━━━━━━━━━━━━━━━━ 22s 7ms/step
WATER R2: 0.8065552742545224

===== TRAINING COND_VOL =====


a:\pfe master\PEG_Python-master\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/20
1486/1486 ━━━━━━━━━━━━━━━━━━━━ 86s 55ms/step - loss: 463.1403 - val_loss: 86.3390
Epoch 2/20
1486/1486 ━━━━━━━━━━━━━━━━━━━━ 84s 57ms/step - loss: 109.7483 - val_loss: 76.3478
Epoch 3/20
1486/1486 ━━━━━━━━━━━━━━━━━━━━ 82s 55ms/step - loss: 98.8436 - val_loss: 78.6862
Epoch 4/20
1486/1486 ━━━━━━━━━━━━━━━━━━━━ 81s 55ms/step - loss: 90.1563 - val_loss: 86.2226
Epoch 5/20
1486/1486 ━━━━━━━━━━━━━━━━━━━━ 82s 55ms/step - loss: 85.4625 - val_loss: 83.4000


3238/3238 ━━━━━━━━━━━━━━━━━━━━ 22s 7ms/step
COND_VOL R2: 0.9337599592232444


In [9]:
# FINAL RESULTS
# =========================================
print("\n===== FINAL RESULTS =====")
print("GAS   R2:", r2_gas)
print("WATER R2:", r2_water)
print("COND  R2:", r2_cond)


===== FINAL RESULTS =====
GAS   R2: 0.9342764656289317
WATER R2: 0.8065552742545224
COND  R2: 0.9337599592232444
